In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("StructuredStreamingLab")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("SparkSession created successfully.")

In [ ]:
sales_data = [
    (1001, "C101", "Laptop", "Electronics", 1, 55000.0, "Mumbai"),
    (1002, "C102", "Mobile", "Electronics", 2, 25000.0, "Pune"),
    (1003, "C103", "Shoes", "Fashion", 2, 3000.0, "Mumbai"),
    (1004, "C104", "Watch", "Fashion", 1, 5000.0, "Delhi"),
    (1005, "C105", "Tablet", "Electronics", 1, 30000.0, "Pune")
]

columns = [
    "order_id",
    "customer_id",
    "product",
    "category",
    "quantity",
    "price",
    "city"
]

batch_df = spark.createDataFrame(
    sales_data,
    columns
)

batch_df.show()

In [ ]:
# Process batch data
from pyspark.sql.functions import col

batch_df = batch_df.withColumn(
    "sales_amount",
    col("quantity") * col("price")
)

batch_df.show()

from pyspark.sql.functions import sum

batch_df.groupBy("city").agg(
    sum("sales_amount").alias("revenue")
).show()

In [ ]:
import os

input_dir = "./tmp/spark_streaming/orders"
checkpoint_dir = "./tmp/spark_streaming/checkpoint"

os.makedirs(input_dir, exist_ok=True)
os.makedirs(checkpoint_dir, exist_ok=True)

In [ ]:
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    DoubleType
)

order_schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product", StringType(), True),
    StructField("category", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("price", DoubleType(), True),
    StructField("city", StringType(), True)
])

In [ ]:
orders_stream = (
    spark.readStream
    .schema(order_schema)
    .option("header", "true")
    .csv(input_dir)
)

In [ ]:
print(orders_stream.isStreaming)

In [ ]:
processed_orders = (
    orders_stream
    .withColumn(
        "sales_amount",
        col("quantity") * col("price")
    )
)

In [ ]:
processed_orders = processed_orders.filter(
    (col("quantity") > 0) &
    (col("price") > 0)
)

In [ ]:
raw_query = (
    processed_orders.writeStream
    .format("console")
    .outputMode("append")
    .option("truncate", "false")
    .option(
        "checkpointLocation",
        "./tmp/spark_streaming/checkpoint_raw"
    )
    .start()
)

In [18]:
print(raw_query.isActive)

True


In [16]:
data1 = """order_id,customer_id,product,category,quantity,price,city
2001,C201,Laptop,Electronics,1,60000,Mumbai
2002,C202,Mobile,Electronics,2,20000,Pune
2003,C203,Shoes,Fashion,2,3500,Mumbai
"""

with open(
    "./tmp/spark_streaming/orders/orders1.csv",
    "w"
) as f:
    f.write(data1)

In [17]:
data2 = """order_id,customer_id,product,category,quantity,price,city
2004,C204,Watch,Fashion,1,8000,Delhi
2005,C205,Tablet,Electronics,2,30000,Pune
2006,C206,Headphones,Electronics,3,2000,Mumbai
"""

with open(
    "./tmp/spark_streaming/orders/orders2.csv",
    "w"
) as f:
    f.write(data2)

In [14]:
city_metrics = (
    processed_orders
    .groupBy("city")
    .agg(
        sum("sales_amount").alias("total_revenue")
    )
)

In [22]:
city_query = (
    city_metrics.writeStream
    .format("console")
    .outputMode("complete")
    .option("truncate", "false")
    .option(
        "checkpointLocation",
        "./tmp/spark_streaming/checkpoint_city"
    )
    .start()
)

In [25]:
city_query.stop()
raw_query.stop()

In [20]:
dashboard_query = (
    city_metrics.writeStream
    .format("memory")
    .queryName("city_dashboard")
    .outputMode("complete")
    .start()
)

In [21]:
spark.sql("""
SELECT *
FROM city_dashboard
ORDER BY total_revenue DESC
""").show()

+------+-------------+
|  city|total_revenue|
+------+-------------+
|  Pune|     100000.0|
|Mumbai|      73000.0|
| Delhi|       8000.0|
+------+-------------+



In [23]:
data3 = """order_id,customer_id,product,category,quantity,price,city
2007,C204,Watch,Fashion,1,18000,Delhi
2008,C205,Tablet,Electronics,2,35000,Pune
2009,C206,Headphones,Electronics,3,4000,Mumbai
"""

with open(
    "./tmp/spark_streaming/orders/orders3.csv",
    "w"
) as f:
    f.write(data3)

In [24]:
spark.sql("""
SELECT *
FROM city_dashboard
ORDER BY total_revenue DESC
""").show()

+------+-------------+
|  city|total_revenue|
+------+-------------+
|  Pune|     170000.0|
|Mumbai|      85000.0|
| Delhi|      26000.0|
+------+-------------+

